# Chest X-Ray Pneumonia — GitHub + Colab (no Drive for code)

1. Push this repo to GitHub (see `GITHUB_SETUP.md`)
2. Set **`GITHUB_USER`** in the next cell
3. **Runtime → GPU** → Run all cells

> Research/education only — not for clinical use.

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Clone from GitHub

In [ ]:
# ============ EDIT THIS ============
GITHUB_USER = "YOUR_USERNAME"  # e.g. "johndoe"
REPO_NAME = "chest_xray_pneumonia_detection"
# ===================================

import os
import sys
from pathlib import Path

if GITHUB_USER == "YOUR_USERNAME":
    raise ValueError("Set GITHUB_USER to your GitHub username in this cell.")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content") / REPO_NAME

if PROJECT_ROOT.exists() and (PROJECT_ROOT / "src" / "train.py").exists():
    print(f"Repo already at {PROJECT_ROOT}")
else:
    if PROJECT_ROOT.exists():
        !rm -rf {PROJECT_ROOT}
    !git clone {REPO_URL} {PROJECT_ROOT}

assert (PROJECT_ROOT / "src" / "train.py").exists(), f"Clone failed: {REPO_URL}"

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "chest_xray"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
BATCH_SIZE = 16

for d in (FIGURES_DIR, MODELS_DIR, DATA_ROOT.parent):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src/ OK:", True)

## 2. Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 3. Download dataset (Kaggle)

Accept rules: [Chest X-Ray Pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
Upload `kaggle.json` when prompted (Kaggle → Account → Create New Token).

In [ ]:
import json
import os

if (DATA_ROOT / "train").exists():
    print(f"Dataset already at {DATA_ROOT}")
else:
    from google.colab import files
    os.makedirs("/root/.kaggle", exist_ok=True)
    if not Path("/root/.kaggle/kaggle.json").exists():
        print("Upload kaggle.json:")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            with open("/root/.kaggle/kaggle.json", "wb") as f:
                f.write(uploaded["kaggle.json"])
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("Downloading ~1.2 GB from Kaggle...")
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {DATA_ROOT.parent} --unzip
    print("Done.")

from src.dataset import get_dataset_stats
import json as _json
print(_json.dumps(get_dataset_stats(DATA_ROOT), indent=2))

## 4. Train (~20–40 min on T4)

In [ ]:
from src.train import train

history = train(
    data_root=DATA_ROOT,
    backbone="resnet50",
    epochs=15,
    batch_size=BATCH_SIZE,
    lr=1e-4,
    output_dir=MODELS_DIR,
    num_workers=2,
)

## 5. Evaluate + heatmaps

In [ ]:
from src.evaluate import run_evaluation
from IPython.display import Image, display

CHECKPOINT = MODELS_DIR / "best_resnet50.pth"
results = run_evaluation(
    checkpoint_path=CHECKPOINT,
    data_root=DATA_ROOT,
    output_dir=FIGURES_DIR,
    split="test",
    generate_cams=True,
    num_cam_samples=8,
    num_failure_cases=4,
)
print(f"PR-AUC: {results['pr_auc']:.4f} | ROC-AUC: {results['roc_auc']:.4f}")

In [ ]:
for name in ["confusion_matrix.png", "precision_recall_curve.png"]:
    p = FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p), width=500))

cam_dir = FIGURES_DIR / "cam_comparisons"
if cam_dir.exists():
    for p in sorted(cam_dir.glob("*.png"))[:3]:
        display(Image(filename=str(p), width=700))

## 6. Save results (before session ends)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/pneumonia_results", "zip", PROJECT_ROOT / "models")
shutil.make_archive("/content/pneumonia_figures", "zip", FIGURES_DIR)
files.download("/content/pneumonia_results.zip")
files.download("/content/pneumonia_figures.zip")
print("Downloaded model + figures to your computer.")